In [0]:
%sql
use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T 
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202510")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# customer360 date
par_month_obj = datetime.strptime(str(par_month), '%Y%m')
next_par_month_obj = par_month_obj + relativedelta(months=1)
cust360_date = int(next_par_month_obj.strftime('%Y%m') + '01')

# debug
display('par_month = ',par_month)
display(' customer360_date = ',cust360_date)


In [0]:
# master data
prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_footfall.parquet'
prep_freq_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_flag_freq.parquet'
prep_feature_360 = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_360_feature.parquet'
profile_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/profile_chula.csv'
date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/day_type_apr_june_26.csv'
nantional_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/country_group_chula.csv'
home_region_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/home_region_chula.csv'

# report path
report_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/proj_3/report1/{par_month}/'

In [0]:
df_date = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(date_path)).select('date','WEEK','day_type_final')

In [0]:
df = spark.read.parquet(prep_path)
     # raw footfall by mall & customertype (is_bmr, is_non_bmr, is_foriegner)

df_freq = spark.read.parquet(prep_freq_path) # raw freq by mall
df_360 = spark.read.parquet(prep_feature_360)
     # raw profile
df_merge = df.join(F.broadcast(df_freq), ['msisdn','name'], 'left')\
    .join(F.broadcast(df_360.drop('is_bmr','is_non_bmr','is_foriegner','a_country_name','demo_tourist_sim_v1_tourist_bin','roaming_flag')), ['msisdn'], 'inner')
df_merge = df_merge\
    .join(df_date, [df_merge.par_day == df_date.date], "left")\
    .withColumnRenamed('par_month','month')
display(df_merge.count())

In [0]:
df_merge = df_merge.withColumn('dwelling_time_hr'
                               , F.when(F.col('actual_total_duration_day').between(0, 900), F.lit('less_than_or_equrl_15'))
  .when(F.col('actual_total_duration_day').between(901, 3600), F.lit('more_than_15_<1hr'))
  .when(F.col('actual_total_duration_day').between(3601, 7200), F.lit('1-2'))
  .when(F.col('actual_total_duration_day').between(7201, 10800), F.lit('2-3'))
  .when(F.col('actual_total_duration_day').between(10801, 14400), F.lit('3-4'))
  .when(F.col('actual_total_duration_day').between(14401, 21600), F.lit('4-6'))
  .otherwise(F.lit('>6'))
)

In [0]:
df_all_columns = df_merge.fillna(0, 
    subset=[
    'weekly_mall_visitors'
    ])\
        .withColumnRenamed('name','mall')\
            .withColumnRenamed('day_type_final','day_type')

In [0]:
columns = ['msisdn'
           , F.col('geog_resident_location_v1_district_en_cat').alias('home_district')
           , F.col('geog_resident_location_v1_sub_district_en_cat').alias('home_subdistrict')]
df_cust360 = spark.read.table('trueanalytics_data.customer360.customer360_snapshot')\
    .filter((F.col('par_day')==cust360_date) & (F.col('activated_flag')=='1'))\
    .select(columns)

In [0]:
df_all_columns = df_all_columns.drop('home_district','home_subdistrict')\
    .join(df_cust360, "msisdn", 'left')


df_all_columns = df_all_columns.withColumn("home_subdistrict", 
    F.when((F.col("home_province")=='bangkok') & (F.col("home_subdistrict")=='chantharakasem'), F.lit('chan kasem'))
    .otherwise(F.col('home_subdistrict'))
)

In [0]:
core_columns = [
    'latitude',
    'longitude',
    'mall',
    'province',
    'district',
    'sub_district',
    'day_type',
    'date']

In [0]:
R1D_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr']
R1D_Non_BMR_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                         'region']
R1D_Foreigner_columns = ['gender','age_range', 'visit_frequency_(day)','weekly_mall_visitors','dwelling_time_hr',
                       'nationality','nationality_group','foreigner_type']


df_bmr_r1d = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+R1D_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_non_bmr_r1d = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R1D_Non_BMR_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))
df_foreigner_r1d = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R1D_Foreigner_columns).agg(F.countDistinct('msisdn').alias('daily_unique_visitor_cnt'))

# save_report
save_to_csv(df_bmr_r1d, report_path+f"report1_bmr_r1d_{par_month}.csv")
save_to_csv(df_non_bmr_r1d, report_path+f"report1_non_bmr_r1d_{par_month}.csv")
save_to_csv(df_foreigner_r1d, report_path+f"report1_foreigner_r1d_{par_month}.csv")

In [0]:
for par_month in ['202601','202602','202603', '202604', '202605', '202606']:
    volume_path = f"dbfs:/Volumes/int-cu-siampiwat/staging/report/proj_3/report1/{par_month}/"
    path_mask = f"dbfs:/Volumes/int-cu-siampiwat/staging/report/proj_3/mask/report1/{par_month}/"
    files = dbutils.fs.ls(volume_path)
    report1_files = [file.name for file in files if file.name.endswith('.csv/')]
    for file in report1_files:
        print()
        df = (spark.read
            .option("header", "true").option("inferSchema", "true")
            .csv(volume_path+file)
        )

        df_marked = df.withColumn("daily_unique_visitor_cnt", 
                                F.when(F.col('daily_unique_visitor_cnt') == 0, F.lit(0))
                                .when(F.col("daily_unique_visitor_cnt") <= 25, 25).otherwise(F.col("daily_unique_visitor_cnt")))

        save_to_csv(df_marked, path_mask+file)